In [22]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-mpnet-base-v2")

# topic_text = """
# war, army, military, conflict, attack, bomb, invasion, battle, soldiers, violence,
# occupation, missile, fighting, defense, siege, casualties, combat, troops, conflict zones
# """
# topic_text = "government, parliament, prime minister, president, elections, policy, law, minister, democracy, legislation"
# topic_text = "economy, inflation, stock, market, business, trade, unemployment, budget, finance, investment, recession"
# topic_text = "protest, health, education, rights, inequality, social, law, welfare, healthcare, crime"
topic_text = "israel, palestine"
# topic_text = "russia, ukraine"

topic_emb = model.encode(topic_text, convert_to_tensor=True)

chunk = "By contrast, the bottom tier of technocrats appears relatively homogeneous – at least in terms of Palestinian identity – though it is reasonable to assume that both the Palestinian Authority and Hamas will exert pressure on each of its members to adopt positions closer to their own. The most difficult task is the demilitarization of Gaza and the disarmament of Hamas. At the Davos meeting, Jared Kushner presented a master plan for dismantling Hamas, demilitarizing the Strip, and rebuilding it, but it remains unclear who would carry out the first two tasks. The “international stabilization force” mentioned in Trump’s plan already has an American commander, but it has no army. Countries that committed to sending forces – such as Azerbaijan and Indonesia – have meanwhile backed out."
chunk_emb = model.encode(chunk, convert_to_tensor=True)

sim = util.cos_sim(chunk_emb, topic_emb)
print(sim)

tensor([[0.3524]])


In [ ]:
import pandas as pd
import re

class Months:
    MONTHS_WORDS1 = {"January": "01", "February": "02", "March": "03", "April": "04", "May": "05", "June": "06", "July": "07", "August": "08", "September": "09", "October": "10", "November": "11", "December": "12"}
    MONTHS_WORDS_ABBR = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "May": "05", "Jun": "06", "Jul": "07", "Aug": "08", "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}
    MONTHS_RUSSIAN = {"января": "01", "февраля": "02", "марта": "03", "апреля": "04","мая": "05", "июня": "06", "июля": "07", "августа": "08","сентября": "09", "октября": "10", "ноября": "11", "декабря": "12"}
    DAYS_TO_REPLACE = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

def clean_date():
    website = "kpru"
    df = pd.read_csv("../data/palestine/alquds_original.csv")
    
    pattern_en = r"\b(" + "|".join(Months.MONTHS_WORDS1.keys()) + r")\b"
    pattern_ru = r"\b(" + "|".join(Months.MONTHS_RUSSIAN.keys()) + r")\b"
    pattern_days = r"\b(" + "|".join(Months.DAYS_TO_REPLACE) + r")\b"
    pattern_abbr = r"\b(" + "|".join(Months.MONTHS_WORDS_ABBR.keys()) + r")\b"


    if website != "liganet":
        if website not in ["kpru", "alquds"]:
            df["date"] = (df["date"].str.replace(r'(?<=\w) (?=\w)', '', regex=True))
        df["date"] = (df["date"]
            .str.replace(pattern_days, "", regex=True, flags=re.IGNORECASE)
            .str.replace(pattern_en, lambda m: Months.MONTHS_WORDS1[m.group(0).title()], regex=True, flags=re.IGNORECASE)
            .str.replace(pattern_ru, lambda m: Months.MONTHS_RUSSIAN[m.group(0).lower()], regex=True, flags=re.IGNORECASE)
            .str.replace(r"[,/]", " ", regex=True)
            .str.replace(r"\s+\d{1,2}\s*:\s*\d{2}.*$", "", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(pattern_abbr, lambda m: Months.MONTHS_WORDS_ABBR[m.group(0).title()], regex=True, flags=re.IGNORECASE)
            .str.strip())
        if website == "kpru":
            df["date"] = (df["date"].str.replace(" ", "-"))

        if website != "kpru":
            df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True).dt.strftime("%d-%m-%Y")
    print(df["date"].dtype)
    print(df["date"])

clean_date()


object
0     20-01-2026
1     20-01-2026
2     20-01-2026
3     20-01-2026
4     20-01-2026
         ...    
95    24-10-2025
96    23-10-2025
97    23-10-2025
98    22-10-2025
99    21-10-2025
Name: date, Length: 100, dtype: object


In [68]:
import matplotlib.pyplot as plt
website = "liganet"
df = pd.read_csv(f"../data/ukraine/{website}_embedded.csv")

filtered_df = df[(df["filter1"] > 0.4) | (df["filter2"] > 0.4) | (df["filter3"] > 0.4)].copy()
filtered_df.to_csv(f"../data/ukraine/{website}_filtered.csv")
print((filtered_df["title"].count() / df["title"].count())*100)

44.565217391304344
